# The Supply Line: Solutions Notebook

This notebook contains complete solutions for both TODOs.

In [ ]:
# Setup
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists('coding-exercises'):
        !cd coding-exercises && git pull
    else:
        !git clone -b week3 https://github.com/eth-bmai-fs26/coding-exercises.git
    os.chdir('coding-exercises/the_supply_line')

sys.path.insert(0, '.')

!pip install google-genai --quiet
print('Setup complete!')

In [ ]:
from supply_line import *
from supply_line.game import SupplyWorld, AgentState
from supply_line.data import OrderCategory, OrderPriority, OrderStatus
from supply_line.agent import TOOLS_DESCRIPTION, parse_action
print('The Supply Line loaded!')

agent, world, tools = create_game()
print()
print(world.queue_summary())

---
## Play the Game Yourself!

In [ ]:
from supply_line.interactive import play_interactive

game = play_interactive()

---
## Solution: TODO 1 — Rule-Based Think Function

Strategy: process orders as a state machine. Each order goes through phases:
1. Read it
2. Lookup KB if needed
3. Apply the correct action (place_order, submit_documents, reject_shipment, etc.)
4. Handle multi-step orders (quality: reject + claim + reorder)
5. Escalate if required (pricing disputes to procurement, big orders to finance)
6. Notify client
7. Close the order

Key insight: the two boss orders (T-001 Aurora, T-020 Regulatory) have **dependencies** — they require
a prepared briefing, which requires resolving prerequisite orders first. The agent must do
**dependency resolution** — skip boss orders whose prerequisites aren't met, handle
prerequisites first, then come back. This is the Supply Line equivalent of BFS pathfinding.

In [ ]:
import re

def _pick_next_order(world):
    """Pick the next order from the queue, skipping those with unmet briefing prerequisites.
    
    This is the dependency resolution step — the Supply Line equivalent of BFS.
    """
    for candidate in world.queue:
        if candidate.status == OrderStatus.RESOLVED:
            continue
        if candidate.requires_briefing and not candidate.briefing_prepared:
            ready, _ = world.check_briefing_ready(candidate.id)
            if not ready:
                continue  # skip — prerequisites not resolved yet
        return candidate
    # All remaining have unmet prerequisites — pick first anyway
    return world.queue[0] if world.queue else None


def _extract_shipment_id(message: str) -> str:
    """Extract shipment ID (e.g. SH-1190) from the order message."""
    match = re.search(r'#(SH-\d+)', message)
    return match.group(1) if match else "SH-0000"


def think_rule_based(agent: AgentState, world: SupplyWorld, history: list[dict]) -> str:
    """Rule-based operations coordinator using a per-order state machine."""
    order = world.active_order

    # --- No active order? Open the next one from the queue ---
    if order is None or order.status == OrderStatus.RESOLVED:
        next_order = _pick_next_order(world)
        if next_order is None:
            return 'ACTION: check_orders()'
        return f'ACTION: read_order(order_id="{next_order.id}")'

    oid = order.id
    cat = order.category

    # --- Phase 1: Handle system alerts immediately ---
    if cat == OrderCategory.SYSTEM_ALERT:
        if not order.action_applied:
            return f'ACTION: acknowledge_alert(alert_id="{oid}")'
        # acknowledge_alert auto-resolves; move on
        next_order = _pick_next_order(world)
        if next_order:
            return f'ACTION: read_order(order_id="{next_order.id}")'
        return 'ACTION: check_orders()'

    # --- Phase 2: Boss fights — dependency-gated ---
    if cat == OrderCategory.LAUNCH:
        ready, missing = world.check_briefing_ready(oid)
        if ready and not order.briefing_prepared:
            return 'ACTION: prepare_launch_briefing()'
        if order.briefing_prepared and not order.action_applied:
            return f'ACTION: authorize_launch(order_id="{oid}")'
        if order.action_applied and not order.notification_sent:
            return 'ACTION: notify_client(template="launch_status")'
        if order.action_applied and order.notification_sent:
            return f'ACTION: close_order(order_id="{oid}")'
        # Prerequisites not met — switch away
        world.active_order = None
        return 'ACTION: check_orders()'

    if cat == OrderCategory.REGULATORY:
        ready, missing = world.check_briefing_ready(oid)
        if ready and not order.briefing_prepared:
            return 'ACTION: prepare_compliance_package()'
        if order.briefing_prepared and not order.action_applied:
            return f'ACTION: submit_compliance(order_id="{oid}")'
        if order.action_applied and not order.notification_sent:
            return 'ACTION: notify_client(template="compliance_status")'
        if order.action_applied and order.notification_sent:
            return f'ACTION: close_order(order_id="{oid}")'
        world.active_order = None
        return 'ACTION: check_orders()'

    # --- Phase 3: Lookup KB if needed ---
    if order.requires_lookup and not order.lookup_done:
        return f'ACTION: lookup_kb(query="{order.lookup_query}")'

    # --- Phase 4: Apply required action ---
    if order.requires_action and not order.action_applied:
        action = order.requires_action

        if action == "place_order":
            if cat == OrderCategory.RUSH:
                return 'ACTION: place_order(supplier="SteelWorks GmbH", sku="as_needed", quantity="500", expedited="true")'
            elif "coating" in order.message.lower() and "MechPro" not in order.message:
                return 'ACTION: place_order(supplier="PureChem Basel", sku="specialty_coating", quantity="200")'
            elif "MechPro" in order.message:
                return 'ACTION: place_order(supplier="MechPro Engineering", sku="spec_v2.1", quantity="1")'
            elif "ChemPure AG" in order.message and cat == OrderCategory.REORDER:
                return 'ACTION: place_order(supplier="ChemPure AG", sku="as_needed", quantity="50")'
            else:
                return 'ACTION: place_order(supplier="SteelWorks GmbH", sku="as_needed", quantity="50")'

        if action == "reject_shipment":
            sid = _extract_shipment_id(order.message)
            return f'ACTION: reject_shipment(shipment_id="{sid}", reason="QC failure contamination")'

        if action == "submit_documents":
            sid = _extract_shipment_id(order.message)
            return f'ACTION: submit_documents(shipment_id="{sid}", doc_type="certificate_of_origin")'

        if action == "update_timeline":
            return f'ACTION: update_timeline(order_id="{oid}", new_eta="5 business days")'

        if action == "file_claim":
            return 'ACTION: file_claim(supplier="SteelWorks GmbH", amount="5000")'

        if action == "compile_records":
            return 'ACTION: compile_records(period="30_days")'

        if action == "acknowledge_alert":
            return f'ACTION: acknowledge_alert(alert_id="{oid}")'

    # --- Phase 4b: Escalate if needed ---
    if order.requires_escalation and order.escalated_to != order.requires_escalation:
        return f'ACTION: escalate(department="{order.requires_escalation}", reason="as per procedure")'

    # --- Phase 4c: Multi-step quality handling (reject + claim + reorder) ---
    if cat == OrderCategory.QUALITY and order.action_applied:
        if not order.claim_filed:
            return 'ACTION: file_claim(supplier="ChemPure AG", amount="25000")'
        recent = [h for h in history if h["role"] == "action"]
        recent_actions = [a["content"] for a in recent[-10:]]
        if not any("place_order" in a for a in recent_actions):
            return 'ACTION: place_order(supplier="PureChem Basel", sku="specialty_coating", quantity="200")'

    # --- Phase 5: Notify client ---
    if order.correct_template and not order.notification_sent:
        return f'ACTION: notify_client(template="{order.correct_template}")'

    # --- Phase 6: Close ---
    return f'ACTION: close_order(order_id="{oid}")'

In [ ]:
result = play_rule_based(think_rule_based, max_turns=100, delay=0.05)
print(f"\nResult: {'Victory!' if result['won'] else 'Not quite...'} | "
      f"Resolved: {result['resolved']} | Quality: {result['quality']}% | "
      f"Tokens: {result['tokens']} | Turns: {result['turns']}")
print(f"Log: {result['log_file']}")

---
## Solution: TODO 2 — LLM-Powered Think Function

Architecture: three layers with clear separation of concerns.

1. **Autopilot** — handles ONLY mechanical tasks: queue management, system alerts, KB lookups, and boss fight state machines (dependency resolution). No decisions.
2. **Memory** — builds structured context for the LLM: order details, KB results, progress, recent history
3. **LLM** — makes the actual decisions: which action to take, which supplier, what parameters, which template, whether to escalate. The LLM reads the order message and KB procedures to figure out what to do.
4. **Guardrails** — catch catastrophic mistakes only: blacklisted suppliers (token loss), premature close (quality), invalid templates, stuck loops. Guardrails don't replace the LLM — they're a safety net.

Key insight: the autopilot does NOT hardcode action parameters (suppliers, shipment IDs, templates).
That's the LLM's job. If the LLM makes a mistake, improve the prompt or add a guardrail — don't
move the decision into the autopilot, or you've just written a second rule-based solution.

In [ ]:
import os
from google import genai

# os.environ['GEMINI_API_KEY'] = 'your-key-here'
try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')
except (ImportError, Exception):
    api_key = os.environ.get('GEMINI_API_KEY')

client = genai.Client(api_key=api_key)
print('Gemini client ready!')

In [ ]:
# ── Layer 1: Autopilot — ONLY queue management, alerts, lookups, boss fights ──
# The autopilot does NOT make decisions about actions, suppliers, or templates.
# Those are the LLM's job.

VALID_TEMPLATES = {
    'reorder_confirmation', 'delay_notification', 'quality_issue',
    'customs_update', 'rush_confirmation', 'launch_status',
    'compliance_status', 'dispute_resolution', 'general_update',
}


def _pick_next_order_llm(world):
    """Pick the next order, skipping those with unmet briefing prerequisites."""
    for candidate in world.queue:
        if candidate.status == OrderStatus.RESOLVED:
            continue
        if candidate.requires_briefing and not candidate.briefing_prepared:
            ready, _ = world.check_briefing_ready(candidate.id)
            if not ready:
                continue
        return candidate
    return world.queue[0] if world.queue else None


def _switch_away_from(order, world):
    """Find another order to work on when the current one is blocked."""
    for candidate in world.queue:
        if candidate.id != order.id and candidate.status != OrderStatus.RESOLVED:
            if candidate.requires_briefing and not candidate.briefing_prepared:
                r, _ = world.check_briefing_ready(candidate.id)
                if not r:
                    continue
            return f'ACTION: read_order(order_id="{candidate.id}")'
    return 'ACTION: check_orders()'


def _auto_handle(agent, world, order):
    """Handle ONLY mechanical tasks. Everything else → LLM."""

    # No active order → pick next
    if order is None or order.status == OrderStatus.RESOLVED:
        next_order = _pick_next_order_llm(world)
        if next_order:
            return f'ACTION: read_order(order_id="{next_order.id}")'
        if world.queue:
            return f'ACTION: read_order(order_id="{world.queue[0].id}")'
        return 'ACTION: check_orders()'

    oid = order.id
    cat = order.category

    # System alerts: trivial, no decision needed
    if cat == OrderCategory.SYSTEM_ALERT:
        if not order.action_applied:
            return f'ACTION: acknowledge_alert(alert_id="{oid}")'
        next_order = _pick_next_order_llm(world)
        if next_order:
            return f'ACTION: read_order(order_id="{next_order.id}")'
        return 'ACTION: check_orders()'

    # KB lookup: always lookup first if required and not done
    if order.requires_lookup and not order.lookup_done:
        query = order.lookup_query or order.category.value
        return f'ACTION: lookup_kb(query="{query}")'

    # Boss fights: dependency resolution (algorithmic planning, not LLM)
    if cat == OrderCategory.LAUNCH:
        ready, _ = world.check_briefing_ready(oid)
        if ready and not order.briefing_prepared:
            return 'ACTION: prepare_launch_briefing()'
        if order.briefing_prepared and not order.action_applied:
            return f'ACTION: authorize_launch(order_id="{oid}")'
        if order.action_applied and not order.notification_sent:
            return 'ACTION: notify_client(template="launch_status")'
        if order.action_applied and order.notification_sent:
            return f'ACTION: close_order(order_id="{oid}")'
        world.active_order = None
        return _switch_away_from(order, world)

    if cat == OrderCategory.REGULATORY:
        ready, _ = world.check_briefing_ready(oid)
        if ready and not order.briefing_prepared:
            return 'ACTION: prepare_compliance_package()'
        if order.briefing_prepared and not order.action_applied:
            return f'ACTION: submit_compliance(order_id="{oid}")'
        if order.action_applied and not order.notification_sent:
            return 'ACTION: notify_client(template="compliance_status")'
        if order.action_applied and order.notification_sent:
            return f'ACTION: close_order(order_id="{oid}")'
        world.active_order = None
        return _switch_away_from(order, world)

    # Everything else → LLM decides
    return None


# ── Layer 2: Memory — structured context for the LLM ─────────────────────

def _build_memory(agent, world, order, history, turns_used):
    """Build context so the LLM can make informed decisions."""
    sections = []

    if order:
        entity = world.get_entity(order.entity_id)
        ename = entity.name if entity else 'Unknown'
        etier = entity.tier.value if entity else 'unknown'

        progress = []
        if order.lookup_done:
            progress.append('KB lookup done')
        if order.escalated_to:
            progress.append(f'Escalated to {order.escalated_to}')
        if order.notification_sent:
            progress.append('Client notified')
        else:
            progress.append('Client NOT YET notified')
        progress_str = ', '.join(progress)

        sections.append(
            f'=== ACTIVE ORDER ===\n'
            f'  {order.id}: {order.subject}\n'
            f'  Entity: {ename} ({etier}) | Priority: {order.priority.value}\n'
            f'  Category: {order.category.value}\n'
            f'  Progress: {progress_str}\n'
            f'  Message: {order.message}'
        )

    # Include the last KB lookup result (full text — the LLM needs this to decide)
    for h in reversed(history):
        if h['role'] == 'result' and 'Knowledge Base' in h.get('content', ''):
            sections.append(f'=== KB PROCEDURE ===\n{h["content"]}')
            break

    turns_left = 100 - turns_used
    sections.append(
        f'=== PERFORMANCE ===\n'
        f'  Fulfilled: {agent.resolved_count}/{agent.WIN_RESOLVED} | '
        f'Quality: {agent.quality_score:.0f}% | '
        f'Tokens: {agent.operations_tokens}/{agent.MAX_TOKENS} | '
        f'Turns left: {turns_left}'
    )

    # Show recent actions so the LLM knows what it already did
    recent = [h for h in history[-10:] if h['role'] in ('action', 'result')]
    if recent:
        action_lines = []
        for h in recent[-8:]:
            prefix = 'You did' if h['role'] == 'action' else 'Result'
            action_lines.append(f'  {prefix}: {h["content"][:200]}')
        sections.append('=== RECENT ACTIONS ===\n' + '\n'.join(action_lines))

    return '\n\n'.join(sections)


# ── Layer 3: System prompt ────────────────────────────────────────────────

SYSTEM_PROMPT = """You are an AI operations coordinator processing supply chain orders.

GOAL: Fulfill orders efficiently with high quality. You have 3 operations tokens —
losing all means game over.

AVAILABLE TOOLS:
{tools}

WORKFLOW FOR EACH ORDER:
1. Read the order (done automatically)
2. Look up KB for procedures (done automatically)
3. Follow the KB procedure — apply the required action(s) with correct parameters
4. Notify the client with the matching template
5. Close the order

TEMPLATE MATCHING (by order category):
  reorder → reorder_confirmation
  delay → delay_notification
  quality → quality_issue
  customs → customs_update
  rush → rush_confirmation
  dispute → dispute_resolution

CRITICAL RULES:
- Follow KB procedures step by step. For multi-step procedures (quality rejections:
  reject_shipment → file_claim → place_order), do ALL steps before notifying.
- Read the order message carefully for specifics: supplier names, shipment IDs, amounts.
- NEVER order from QuickShip or BargainParts (blacklisted — costs a token!).
- Rush orders under 1000 units → place_order with expedited="true". Do NOT escalate.
- Pricing disputes > 5% → escalate to procurement.
- Order value > EUR 50,000 → escalate to finance.
- Always notify_client with the right template before close_order.
- When all steps are done: close the order. Don't loop on check_orders.

OUTPUT FORMAT:

REASONING: [What this order needs and what step to do next]
ACTION: tool_name(param1="value1", param2="value2")
"""


# ── Layer 4: The think function ───────────────────────────────────────────

def think_llm(agent: AgentState, world: SupplyWorld, history: list[dict]) -> str:
    """LLM-powered coordinator: autopilot for trivial/boss, LLM for decisions, guardrails for safety."""
    order = world.active_order
    turns_used = len([h for h in history if h['role'] == 'action'])

    # ── Autopilot: queue management, alerts, lookups, boss fights only ──
    auto = _auto_handle(agent, world, order)
    if auto:
        return auto

    # ── LLM: make the actual decision ──
    memory = _build_memory(agent, world, order, history, turns_used)
    system = SYSTEM_PROMPT.format(tools=TOOLS_DESCRIPTION)

    user_msg = f"""{memory}

Based on the order details and KB procedure, what is the next step?
If you've completed all required actions, notify the client then close."""

    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=user_msg,
            config=genai.types.GenerateContentConfig(
                system_instruction=system,
                max_output_tokens=500,
                temperature=0.2,
            ),
        )
        text = response.text
    except Exception:
        # API error fallback: try to close gracefully
        if order and order.notification_sent:
            return f'ACTION: close_order(order_id="{order.id}")'
        if order:
            return 'ACTION: notify_client(template="general_update")'
        return 'ACTION: check_orders()'

    # ── Guardrails: catch catastrophic mistakes only ──
    tool_name, args = parse_action(text)

    # Guard: block blacklisted suppliers (costs a token!)
    if tool_name == 'place_order':
        supplier = args.get('supplier', '').lower()
        if any(b in supplier for b in ['quickship', 'bargainparts']):
            return 'ACTION: check_alternatives()'

    # Guard: block stuck loops (LLM outputting check_orders with active order)
    if tool_name in ('check_orders', 'check_stats') and order:
        if order.notification_sent:
            return f'ACTION: close_order(order_id="{order.id}")'
        return f'ACTION: read_order(order_id="{order.id}")'

    # Guard: don't close without notifying
    if tool_name == 'close_order' and order:
        if not order.notification_sent and order.correct_template:
            return f'ACTION: notify_client(template="{order.correct_template}")'

    # Guard: invalid template name
    if tool_name == 'notify_client':
        template = args.get('template', '')
        if template not in VALID_TEMPLATES:
            return 'ACTION: notify_client(template="general_update")'

    # Guard: block boss fight shortcuts
    if tool_name == 'escalate' and order:
        if order.requires_briefing and not order.briefing_prepared:
            world.active_order = None
            return _switch_away_from(order, world)

    return text

In [ ]:
result = play_with_llm(
    think_fn=lambda agent, world, history: think_llm(agent, world, history),
    max_turns=100,
    delay=0.5,
)
print(f"\nResult: {'Victory!' if result['won'] else 'Not quite...'} | "
      f"Resolved: {result['resolved']} | Quality: {result['quality']}% | "
      f"Tokens: {result['tokens']} | Turns: {result['turns']}")
print(f"Log: {result['log_file']}")

In [ ]:
# Download the game log
try:
    from google.colab import files
    files.download(result['log_file'])
except ImportError:
    print(f"Log file: {result['log_file']}")
    print('Open the file to inspect your agent\'s decisions turn by turn.')